In [1]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
# Parameters
max_features = 10000  # Number of words to consider as features
maxlen = 200          # Cut texts after this number of words (among top max_features most common words)
embedding_dim = 128   # Embedding dimensionality
batch_size = 32
epochs = 5

In [3]:
# Load the IMDB dataset
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)


In [4]:
# Pad sequences to ensure consistent length
x_train = sequence.pad_sequences(x_train, maxlen=maxlen)
x_test = sequence.pad_sequences(x_test, maxlen=maxlen)

In [5]:
# Build the model
model = Sequential([
    Embedding(max_features, embedding_dim, input_length=maxlen),
    LSTM(128, dropout=0.2, recurrent_dropout=0.2), # Add dropout to prevent overfitting.
    Dense(1, activation='sigmoid')  # Binary classification (positive/negative)
])

2025-04-11 21:39:46.226320: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2025-04-11 21:39:46.226353: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2025-04-11 21:39:46.226362: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2025-04-11 21:39:46.226559: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:303] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-04-11 21:39:46.226742: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:269] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [6]:
# Compile the model
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [7]:
# Early stopping to prevent overfitting
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)


In [ ]:
# Train the model
history = model.fit(x_train, y_train,
                    epochs=epochs,
                    batch_size=batch_size,
                    validation_split=0.2,
                    callbacks=[early_stopping])


Epoch 1/5


2025-04-11 21:39:47.627054: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


323/625 [==============>...............] - ETA: 8:14:25 - loss: 0.5134 - accuracy: 0.7426

In [ ]:
# Evaluate the model
loss, accuracy = model.evaluate(x_test, y_test, batch_size=batch_size)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

In [ ]:
# Example of how to get the word embeddings.
embedding_layer = model.get_layer(index=0) # get the embedding layer
embedding_weights = embedding_layer.get_weights()[0] # get the weights.
print(f"Embedding weights shape: {embedding_weights.shape}")